# Patrón de Comportamiento: Observer — Cafetería

**Dominio propio:** cuando un pedido cambia de estado (recibido → preparando → listo), deben reaccionar varias partes: la pantalla de la cocina, la app del cliente y el panel de estadísticas.

## Introducción — qué problema resuelve
Un `Pedido` cambia de estado a lo largo de su vida. Cada cambio interesa a varios consumidores que reaccionan distinto: la pantalla de cocina muestra el pedido, la app del cliente le manda una notificación, y el panel de estadísticas cuenta pedidos listos. El pedido **no debería conocer** a cada consumidor concreto ni su lógica.

El patrón **Observer** define una relación uno-a-muchos: cuando el pedido (sujeto) cambia de estado, notifica automáticamente a todos sus observadores suscritos.

## Sin patrón (el problema es evidente)

`Pedido` tiene cableada dentro la notificación a cada consumidor. Para agregar un consumidor nuevo (o quitar uno) hay que **modificar** el método `cambiar_estado`.

In [1]:
class Pedido:
    def __init__(self, numero: int) -> None:
        self.numero = numero
        self.estado = "recibido"

    def cambiar_estado(self, nuevo_estado: str) -> None:
        self.estado = nuevo_estado
        # Logica de notificacion cableada y acoplada:
        print(f"[Cocina] Pedido #{self.numero} -> {nuevo_estado}")
        print(f"[App cliente] Tu pedido #{self.numero} ahora esta '{nuevo_estado}'")
        # Para agregar el panel de estadisticas hay que EDITAR este metodo


pedido = Pedido(101)
pedido.cambiar_estado("preparando")
pedido.cambiar_estado("listo")

[Cocina] Pedido #101 -> preparando
[App cliente] Tu pedido #101 ahora esta 'preparando'
[Cocina] Pedido #101 -> listo
[App cliente] Tu pedido #101 ahora esta 'listo'


### Problema visible
La lógica de cada consumidor vive dentro de `Pedido`. Agregar el panel de estadísticas obliga a modificar `cambiar_estado`, y el `Pedido` queda acoplado a cocina, app, etc. Viola el principio abierto/cerrado.

## Con patrón Observer (problema resuelto)

Defino una interfaz `ObservadorPedido`. El `Pedido` (sujeto) mantiene una lista de observadores y los notifica; cada observador concreto reacciona a su manera. Agregar consumidores no toca al `Pedido`.

In [2]:
from abc import ABC, abstractmethod

class ObservadorPedido(ABC):
    @abstractmethod
    def actualizar(self, numero: int, estado: str) -> None: ...


class PantallaCocina(ObservadorPedido):
    def actualizar(self, numero: int, estado: str) -> None:
        print(f"[Cocina] Pedido #{numero} -> {estado}")


class AppCliente(ObservadorPedido):
    def __init__(self, cliente: str) -> None:
        self.cliente = cliente

    def actualizar(self, numero: int, estado: str) -> None:
        print(f"[App {self.cliente}] Tu pedido #{numero} ahora esta '{estado}'")


class PanelEstadisticas(ObservadorPedido):
    def __init__(self) -> None:
        self.listos = 0

    def actualizar(self, numero: int, estado: str) -> None:
        if estado == "listo":
            self.listos += 1
            print(f"[Stats] Pedidos listos hoy: {self.listos}")


class Pedido:
    """Sujeto: no conoce los tipos concretos de observador."""
    def __init__(self, numero: int) -> None:
        self.numero = numero
        self.estado = "recibido"
        self._observadores: list[ObservadorPedido] = []

    def suscribir(self, obs: ObservadorPedido) -> None:
        self._observadores.append(obs)

    def cambiar_estado(self, nuevo_estado: str) -> None:
        self.estado = nuevo_estado
        for obs in self._observadores:
            obs.actualizar(self.numero, nuevo_estado)


pedido = Pedido(101)
pedido.suscribir(PantallaCocina())
pedido.suscribir(AppCliente("Ana"))
pedido.suscribir(PanelEstadisticas())

pedido.cambiar_estado("preparando")
pedido.cambiar_estado("listo")

[Cocina] Pedido #101 -> preparando
[App Ana] Tu pedido #101 ahora esta 'preparando'
[Cocina] Pedido #101 -> listo
[App Ana] Tu pedido #101 ahora esta 'listo'
[Stats] Pedidos listos hoy: 1


### Resultado
El `Pedido` solo recorre su lista de observadores y llama `actualizar`. Agregué `PanelEstadisticas` **sin modificar** `Pedido`, y puedo suscribir/quitar consumidores en tiempo de ejecución.

## Diagrama UML (clases reales de este ejemplo)
```plantuml
@startuml
abstract class ObservadorPedido {
    + actualizar(numero, estado)
}
class PantallaCocina
class AppCliente
class PanelEstadisticas
ObservadorPedido <|-- PantallaCocina
ObservadorPedido <|-- AppCliente
ObservadorPedido <|-- PanelEstadisticas
class Pedido {
    - numero: int
    - estado: str
    - _observadores: list
    + suscribir(obs)
    + cambiar_estado(nuevo_estado)
}
Pedido --> ObservadorPedido : notifica
@enduml
```

## ¿Por qué Observer y no otro patrón de comportamiento?

El problema es que **un cambio de estado en un objeto debe disparar reacciones en varios objetos independientes**, que pueden entrar y salir dinámicamente, sin que el emisor los conozca. Eso es una relación uno-a-muchos con notificación automática: la definición exacta de **Observer**.

No es Strategy (ahí se intercambia **un** algoritmo dentro de un contexto, no se notifica a muchos), ni State (que encapsula el comportamiento según el estado, no la difusión del cambio), ni Mediator (que centraliza la comunicación entre muchos objetos que se hablan entre sí, no un sujeto que difunde a suscriptores). Como lo que necesito es **difundir** un evento a N consumidores desacoplados, Observer es el patrón correcto.